## Model Scoring API Testing and Coverage Report

This notebook prepares and tests a spam classification scoring service. It uploads the chosen trained model and the application files (`app.py`, `score.py`, and `test.py`). Since the testing environment is Google Colab, the original test file is modified and written as `test_colab.py` to ensure compatibility with the Colab runtime.

The notebook then executes the test suite using `pytest`, which includes both unit tests for the scoring function and integration tests for the Flask API endpoint. Finally, a coverage report is generated and stored in `coverage.txt` to evaluate how much of the codebase is exercised by the tests.

In [1]:
!pip install -q flask pytest pytest-cov requests
print("All libraries installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.1/254.1 kB 7.3 MB/s eta 0:00:00
All libraries installed.


When the picker opens, select **all three** at once:
- `score.py`
- `app.py`
- `test.py`
- `best_model.joblib`

In [3]:
from google.colab import files
uploaded = files.upload()

import os
for f in uploaded:
    print(f"Uploaded: {f}  ({os.path.getsize(f)} bytes)")

Saving app.py to app (1).py
Saving test.py to test (1).py
Saving score.py to score (1).py
Saving best_model.joblib to best_model.joblib
Uploaded: app (1).py  (873 bytes)
Uploaded: test (1).py  (5378 bytes)
Uploaded: score (1).py  (1115 bytes)
Uploaded: best_model.joblib  (196000 bytes)


### Quick sanity check on score.py

In [4]:
import joblib
from score import score

model = joblib.load("best_model.joblib")

spam_pred, spam_prop = score("URGENT! You won a FREE prize! Call 09061743811", model, 0.5)
ham_pred,  ham_prop  = score("Hey, are you coming to dinner tonight?",          model, 0.5)

print(f"SPAM text  -> prediction={spam_pred}, propensity={spam_prop:.4f}")
print(f"HAM  text  -> prediction={ham_pred},  propensity={ham_prop:.4f}")
print()
print("Expected: SPAM=True (high propensity), HAM=False (low propensity)")

SPAM text  -> prediction=True, propensity=0.6788
HAM  text  -> prediction=False,  propensity=0.1644

Expected: SPAM=True (high propensity), HAM=False (low propensity)


### Write Colab-compatible test.py
> The original `test.py` uses `subprocess` to start Flask — unreliable in Colab.
> This version uses a **background thread** instead. All 15 tests are identical.

In [7]:
test_lines = [
    "import os, time, threading, joblib, requests, pytest\n",
    "from score import score\n",
    "from app import app as flask_app\n",
    "\n",
    "MODEL_PATH = 'best_model.joblib'\n",
    "\n",
    "@pytest.fixture(scope='module')\n",
    "def model():\n",
    "    return joblib.load(MODEL_PATH)\n",
    "\n",
    "@pytest.fixture(scope='module')\n",
    "def flask_url():\n",
    "    t = threading.Thread(\n",
    "        target=lambda: flask_app.run(host='0.0.0.0', port=5000, use_reloader=False),\n",
    "        daemon=True)\n",
    "    t.start()\n",
    "    time.sleep(2)\n",
    "    return 'http://127.0.0.1:5000'\n",
    "\n",
    "class TestScore:\n",
    "\n",
    "    def test_smoke(self, model):\n",
    "        assert score('Free prize!', model, 0.5) is not None\n",
    "\n",
    "    def test_output_format(self, model):\n",
    "        p, s = score('Hello there', model, 0.5)\n",
    "        assert isinstance(p, bool) and isinstance(s, float)\n",
    "\n",
    "    def test_prediction_is_binary(self, model):\n",
    "        p, _ = score('Win cash now!', model, 0.5)\n",
    "        assert p in (True, False)\n",
    "\n",
    "    def test_propensity_range(self, model):\n",
    "        _, s = score('Call me when free', model, 0.5)\n",
    "        assert 0.0 <= s <= 1.0\n",
    "\n",
    "    def test_threshold_zero_always_spam(self, model):\n",
    "        p, _ = score('See you tomorrow', model, 0.0)\n",
    "        assert p is True\n",
    "\n",
    "    def test_threshold_one_always_ham(self, model):\n",
    "        p, _ = score('FREE CASH WIN NOW CALL 0800', model, 1.0)\n",
    "        assert p is False\n",
    "\n",
    "    def test_obvious_spam(self, model):\n",
    "        p, _ = score('URGENT you won FREE membership call 09061743811', model, 0.5)\n",
    "        assert p is True\n",
    "\n",
    "    def test_obvious_ham(self, model):\n",
    "        p, _ = score('Hey are you coming to dinner tonight?', model, 0.5)\n",
    "        assert p is False\n",
    "\n",
    "class TestFlask:\n",
    "\n",
    "    def test_flask_smoke(self, flask_url):\n",
    "        r = requests.post(flask_url + '/score', json={'text': 'Hello'})\n",
    "        assert r.status_code == 200\n",
    "\n",
    "    def test_flask_response_format(self, flask_url):\n",
    "        d = requests.post(flask_url + '/score', json={'text': 'Win free iPhone!'}).json()\n",
    "        assert 'prediction' in d and 'propensity' in d\n",
    "\n",
    "    def test_flask_prediction_type(self, flask_url):\n",
    "        d = requests.post(flask_url + '/score', json={'text': 'Are you free tonight?'}).json()\n",
    "        assert isinstance(d['prediction'], bool) and isinstance(d['propensity'], float)\n",
    "\n",
    "    def test_flask_spam_detection(self, flask_url):\n",
    "        r = requests.post(flask_url + '/score', json={'text': 'Congratulations cash prize call now!'})\n",
    "        assert r.json()['prediction'] is True\n",
    "\n",
    "    def test_flask_ham_detection(self, flask_url):\n",
    "        r = requests.post(flask_url + '/score', json={'text': 'See you at the library'})\n",
    "        assert r.json()['prediction'] is False\n",
    "\n",
    "    def test_flask_custom_threshold(self, flask_url):\n",
    "        r = requests.post(flask_url + '/score',\n",
    "                          json={'text': 'FREE prize CALL NOW', 'threshold': 1.0})\n",
    "        assert r.json()['prediction'] is False\n",
    "\n",
    "    def test_flask_missing_text_returns_400(self, flask_url):\n",
    "        r = requests.post(flask_url + '/score', json={'threshold': 0.5})\n",
    "        assert r.status_code == 400\n",
]

with open("test_colab.py", "w") as f:
    f.writelines(test_lines)

print("test_colab.py written successfully.")

test.py written successfully.


###Run all tests and save coverage.txt

In [8]:
!pytest test_colab.py -v \
    --cov=score --cov=app \
    --cov-report=term-missing \
    2>&1 | tee coverage.txt

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: cov-7.0.0, anyio-4.12.1, langsmith-0.7.7, typeguard-4.5.1
collecting ... collected 15 items

test.py::TestScore::test_smoke PASSED                                    [  6%]
test.py::TestScore::test_output_format PASSED                            [ 13%]
test.py::TestScore::test_prediction_is_binary PASSED                     [ 20%]
test.py::TestScore::test_propensity_range PASSED                         [ 26%]
test.py::TestScore::test_threshold_zero_always_spam PASSED               [ 33%]
test.py::TestScore::test_threshold_one_always_ham PASSED                 [ 40%]
test.py::TestScore::test_obvious_spam PASSED                             [ 46%]
test.py::TestScore::test_obvious_ham PASSED                              [ 53%]
test.py::TestFlask::test_flask_smoke PASSED           

###Download coverage.txt

In [9]:
from google.colab import files
files.download("coverage.txt")
print("coverage.txt downloaded to your device.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

coverage.txt downloaded to your device.


## Interpretation of Test Output

The test results indicate that all **15 tests passed successfully**, covering both the model scoring logic and the Flask API service.

### Unit Testing (Score Function)

The tests for the scoring function confirm that:

* The function executes without crashing (smoke test).
* Output types and formats are correct.
* Predictions are binary (0 or 1).
* Propensity scores fall within the valid probability range ([0,1]).
* Edge cases behave as expected:

  * When the threshold is set to 0, the prediction always returns spam.
  * When the threshold is set to 1, the prediction always returns non-spam.
* Typical inputs such as obvious spam and non-spam texts are correctly classified.

These results suggest that the scoring logic is implemented correctly and handles both normal and edge cases.

### Integration Testing (Flask API)

The Flask integration tests verify that:

* The API endpoint launches successfully.
* The `/score` endpoint responds correctly to POST requests.
* The returned JSON structure contains both `prediction` and `propensity`.
* Prediction values and probability ranges are valid.
* The API correctly classifies typical spam and non-spam inputs.
* Invalid requests (such as missing text input) are handled properly with appropriate error responses.

This confirms that the Flask service is correctly integrated with the scoring function and responds as expected.

### Code Coverage

The coverage report shows:

| File               | Coverage |
| ------------------ | -------- |
| app.py             | 94%      |
| score.py           | 79%      |
| **Total Coverage** | **88%**  |

An overall coverage of **88%** indicates that the majority of the codebase is exercised by the tests. Only a few lines remain untested, which may correspond to rarely triggered conditions or fallback logic.

Overall, the testing results demonstrate that the scoring function and Flask API are functioning correctly and are well-covered by the test suite.
